# 🌿 Leva-TTS — Evaluation on Colab T4

Reproduce the performance metrics on a **free T4 GPU**:
**RTF**, **TTFA**, **peak VRAM**, **CER/WER** (Whisper round-trip) and **UTMOS** (reference-free MOS).

This clones the repo (the eval script lives there) and runs `scripts/evaluate.py`.

👉 Use a **T4 GPU** runtime. A short sentence set keeps it within Colab limits.

## Clone the repo + install

> The eval script lives in the repo. We install the maintained **`coqui-tts`**
> fork for the XTTS engine (the old `TTS` won't install on Colab's Python), then
> the repo itself, plus Whisper + UTMOS deps for the quality metrics.
> No kernel restart needed.

In [ ]:
import sys
print('Kernel Python:', sys.version.split()[0])

!git clone -q https://github.com/MohammedAly22/Leva-TTS.git
%cd /content/Leva-TTS

# Engine fork (provides the `TTS`/XTTS modules on Python 3.12)
!{sys.executable} -m pip install -q "coqui-tts[codec]==0.27.5" "transformers<5" "numpy>=2"   # keeps NumPy 2, no restart
# Replace the original `coqpit` (Colab preinstalls it) with coqui's fork `coqpit-config`
!{sys.executable} -m pip uninstall -y -q coqpit coqpit-config
!{sys.executable} -m pip install -q --force-reinstall --no-deps coqpit-config
# The repo (editable) without re-pulling the old TTS
!{sys.executable} -m pip install -q --no-deps -e .
!{sys.executable} -m pip install -q soundfile librosa huggingface_hub rich pandas sympy "numpy>=2"
# Evaluation extras
!{sys.executable} -m pip install -q faster-whisper jiwer
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null
import importlib; importlib.invalidate_caches()
import leva_tts, TTS; from TTS.tts.models.xtts import Xtts
print('✅ ready · leva_tts', leva_tts.__version__, '· TTS', TTS.__version__)

## Check the GPU

In [ ]:
# Verify a GPU runtime is active (Runtime ▸ Change runtime type ▸ T4 GPU)
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "⚠️ No GPU — switch the runtime to a GPU!")

## Download the checkpoint

In [ ]:
import leva_tts
leva_tts.download_model("./checkpoints")
print("checkpoint ready")

## Run the evaluation

Runs the full built-in `TEST_SENTENCES` set (Levantine / code-switch / English).
On a T4 this takes a few minutes (Whisper + UTMOS download on first run).

- Add `--optimize` to benchmark the TF32 + `torch.compile` path.
- Add `--no-asr` and/or `--no-utmos` to skip the slow quality metrics and
  measure only latency / RTF / VRAM.

In [ ]:
!python scripts/evaluate.py \
    --checkpoint checkpoints \
    --speaker Mohamed \
    --tag colab_t4

## Inspect the report

In [ ]:
import glob, json
rep = sorted(glob.glob("eval_results/*colab_t4*.json")) or sorted(glob.glob("eval_results/*.json"))
data = json.load(open(rep[-1]))
print(json.dumps(data.get("summary", data), indent=2, ensure_ascii=False)[:2000])

> ⏱️ T4 is slower than the H100 used in the README, so absolute RTF/TTFA will be higher,
> but the **relative** quality metrics (CER/WER/UTMOS) are representative.